In [ ]:
import sqlite3
import pandas as pd

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)

In [2]:

tables = pd.read_sql_query(
"""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""",
conn
)

tables

,name
0,demo
1,drug
2,faers_severity_dataset
3,fluorouracil_filtered
4,indi
5,outc
6,reac
7,rpsr
8,ther


In [3]:
conn.execute("DROP TABLE IF EXISTS fluorouracil_filtered;")
conn.execute("""
CREATE TABLE fluorouracil_filtered AS
SELECT *
FROM drug
WHERE drugname IN (
    SELECT drugname
    FROM drug
    WHERE drugname LIKE '%FLUOROURACIL%'
    GROUP BY drugname
    HAVING COUNT(*) > 10
);
""")


In [5]:
FU_df_filtered = pd.read_sql_query("""
SELECT *
FROM fluorouracil_filtered
""", conn)

FU_df_filtered.shape

(13211, 21)

In [ ]:

# From this cell onward, classes downsampled to balance the dataset for modeling. The most common drug, FLUOROURACIL, 
# has 1,000+ reports (see 5_FU_explore.ipynb), while the least sufficiently sampled common drug, FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN, has only 55 reports. 
# To ensure a balanced dataset for modeling, we will downsample the FLUOROURACIL class to 55 reports, matching the size of the least common class with sufficient samples.


# Downsample the data to balance the classes for modeling. Downsample FLUOROURACIL to the size of FLUOROURACIL\IRINOTECAN\LEUCOVORIN\OXALIPLATIN (55 reports) to ensure balance.

# count reports per drug
counts = FU_df_filtered['drugname'].value_counts()

# keep only groups with >=55 reports
valid_drugs = counts[counts >= 55].index
filtered_df = FU_df_filtered[FU_df_filtered['drugname'].isin(valid_drugs)]

# downsample each group to 55
balanced_df = (
    filtered_df
    .groupby("drugname", group_keys=False)
    .apply(lambda x: x.sample(55, random_state=42))
    .reset_index(drop=True)
)

balanced_df.shape

(275, 20)

## Unsupervised Toxicity Phenotype Clustering (UMAP + HDBSCAN)

Are there distinct types of 5-FU patients based on *what combination of side effects they report*? The pipeline below builds a sparse binary feature matrix (report x top-100 reaction), projects it into 2D with UMAP, clusters the projection with HDBSCAN, and characterizes each cluster by the reactions that define it. Demographics and regimen are used only as post-hoc color overlays, not as input features, so age/sex/regimen do not dominate the distance metric.

**Prerequisites:** `umap-learn` and `hdbscan`. Install once with the commented-out cell below.

In [ ]:
# Install once if the imports below fail. Comment out afterwards.
# %pip install umap-learn hdbscan

In [ ]:
import numpy as np

# 1. Which reactions are the top 100 in the 5-FU cohort?
top_reactions = pd.read_sql_query("""
    SELECT r.pt, COUNT(DISTINCT r.primaryid) AS n
    FROM reac r
    JOIN (SELECT DISTINCT primaryid FROM fluorouracil_analysis) f
        ON r.primaryid = f.primaryid
    GROUP BY r.pt
    ORDER BY n DESC
    LIMIT 100
""", conn)['pt'].tolist()

# 2. Pull (primaryid, reaction) pairs restricted to those 100 reactions.
placeholders = ','.join(['?'] * len(top_reactions))
report_reactions = pd.read_sql_query(f"""
    SELECT DISTINCT r.primaryid, r.pt
    FROM reac r
    JOIN (SELECT DISTINCT primaryid FROM fluorouracil_analysis) f
        ON r.primaryid = f.primaryid
    WHERE r.pt IN ({placeholders})
""", conn, params=top_reactions)

print(f"{report_reactions['primaryid'].nunique():,} unique reports x {len(top_reactions)} reactions")

In [ ]:
# Build the sparse binary feature matrix.
# Rows = primaryid, columns = reaction pt, cells = 1 if present else 0.
X_df = (
    report_reactions
    .assign(present=1)
    .pivot_table(index='primaryid', columns='pt', values='present', fill_value=0)
)
print(f"Feature matrix: {X_df.shape}  (reports x reactions)")
X_df.head(3)

In [ ]:
# Demographics + regimen label per report, kept SEPARATE from the UMAP features.
# These become color overlays after clustering.

oxa_set = set(pd.read_sql_query(
    "SELECT DISTINCT primaryid FROM drug WHERE drugname LIKE '%OXALIPLATIN%'", conn
)['primaryid'])
iri_set = set(pd.read_sql_query(
    "SELECT DISTINCT primaryid FROM drug WHERE drugname LIKE '%IRINOTECAN%'", conn
)['primaryid'])

def classify_regimen(pid):
    in_oxa, in_iri = pid in oxa_set, pid in iri_set
    if in_oxa and in_iri: return 'FOLFIRINOX'
    elif in_oxa:          return 'FOLFOX'
    elif in_iri:          return 'FOLFIRI'
    else:                 return 'Monotherapy'

demo_df = pd.read_sql_query("""
    SELECT DISTINCT d.primaryid, d.sex, d.age
    FROM demo d
    JOIN (SELECT DISTINCT primaryid FROM fluorouracil_analysis) f
        ON d.primaryid = f.primaryid
""", conn).set_index('primaryid')
demo_df['regimen'] = demo_df.index.map(classify_regimen)
demo_df.head()

In [ ]:
import umap

# UMAP: project the 100-D reaction matrix down to 2D for visualization + clustering.
# n_neighbors  = size of local neighborhood UMAP tries to preserve (smaller = tighter clusters).
# min_dist     = how tightly points can pack in the projection.
# random_state = reproducibility.
reducer   = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_df.values)   # shape: (n_reports, 2)
print("UMAP embedding shape:", embedding.shape)

In [ ]:
import hdbscan

# HDBSCAN: density-based clustering, no k to guess.
# min_cluster_size = smallest group we're willing to call a cluster.
# min_samples      = how conservative HDBSCAN is about noise assignment.
# Label -1 means "noise" (didn't cleanly belong to any cluster).
clusterer      = hdbscan.HDBSCAN(min_cluster_size=50, min_samples=10)
cluster_labels = clusterer.fit_predict(embedding)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()
print(f"Found {n_clusters} clusters, {n_noise:,} noise points ({n_noise / len(cluster_labels):.1%})")

In [ ]:
# Characterize each cluster by the reactions that DEFINE it (over-represented vs cohort).
# 'lift' = cluster's reaction rate / cohort's reaction rate.
# lift = 3.0 means the reaction is 3x more common in that cluster than in the overall 5-FU cohort.
X_labeled = X_df.copy()
X_labeled['cluster'] = cluster_labels

cohort_rates = X_df.mean(axis=0)

for c in sorted(set(cluster_labels)):
    if c == -1:
        continue
    cluster_rows  = X_labeled[X_labeled['cluster'] == c].drop(columns='cluster')
    cluster_rates = cluster_rows.mean(axis=0)
    lift          = (cluster_rates / cohort_rates).sort_values(ascending=False)
    print(f"\n=== Cluster {c}  ({len(cluster_rows):,} reports) ===")
    print(lift.head(5).round(2).to_string())

In [ ]:
import matplotlib.pyplot as plt

# Combine the UMAP embedding with demographics for four side-by-side views.
plot_df = pd.DataFrame({
    'umap_1':   embedding[:, 0],
    'umap_2':   embedding[:, 1],
    'cluster':  cluster_labels,
    'primaryid': X_df.index,
}).merge(demo_df.reset_index(), on='primaryid', how='left')

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Panel 1: HDBSCAN cluster
ax = axes[0, 0]
ax.scatter(plot_df['umap_1'], plot_df['umap_2'],
           c=plot_df['cluster'], s=4, cmap='tab10', alpha=0.6)
ax.set_title('HDBSCAN cluster')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# Panel 2: sex
ax = axes[0, 1]
for sex, sub in plot_df.dropna(subset=['sex']).groupby('sex'):
    ax.scatter(sub['umap_1'], sub['umap_2'], s=4, alpha=0.5, label=sex)
ax.set_title('Sex'); ax.legend()
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# Panel 3: regimen
ax = axes[1, 0]
for reg, sub in plot_df.dropna(subset=['regimen']).groupby('regimen'):
    ax.scatter(sub['umap_1'], sub['umap_2'], s=4, alpha=0.5, label=reg)
ax.set_title('Regimen'); ax.legend()
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# Panel 4: age (continuous colormap)
ax   = axes[1, 1]
mask = plot_df['age'].between(18, 100)
sc   = ax.scatter(plot_df.loc[mask, 'umap_1'], plot_df.loc[mask, 'umap_2'],
                  c=plot_df.loc[mask, 'age'], s=4, alpha=0.6, cmap='viridis')
plt.colorbar(sc, ax=ax, label='Age (years)')
ax.set_title('Age')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

plt.tight_layout()
plt.show()